# Episode 12: The Artificial Neuron and Activation Functions
<a href="https://colab.research.google.com/github/Kinvecta/how-llms-work/blob/main/notebooks/Episode%2012%20-%20Neuron%20and%20Activations.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**Companion notebook** for the series *How Large Language Models Work: From Math to Transformers*.

This notebook reproduces the Math Lab numbers from Episode 12, one step per section. The first six steps use only `numpy` and the standard library (already installed in Colab), so choose **Runtime, Run all**. The last section downloads GPT-2 small's published weights (548 MB, once) to count the feed-forward parameters; skip it on a slow connection.

Covers: a neuron as a weighted vote plus a bias and a bend; two affine layers collapsing into one; a ReLU between them breaking the collapse (the midpoint test); the values of ReLU, GELU and the sigmoid σ; GELU against its two approximations; the slope of σ; and the feed-forward sub-layer in GPT-2 small.

Every value that appears in the episode is checked with an `assert`. If a cell fails, a number in the script and the math have drifted apart. All weights here are hand-chosen and illustrative.

In [1]:
import math
import numpy as np

def Phi(z):                       # standard normal CDF: chance a bell-curve number is at most z
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

def sigma(z):                     # the sigmoid, sigma(z) = 1 / (1 + exp(-z))
    return 1.0 / (1.0 + math.exp(-z))

def relu(z):                      # ReLU(z) = max(0, z)
    return max(0.0, z)

def gelu(z):                      # GELU(z) = z * Phi(z)
    return z * Phi(z)

def gelu_tanh(z):                 # tanh form, as in the GELU paper (v5) and OpenAI's gpt-2 model.py
    return 0.5 * z * (1.0 + math.tanh(math.sqrt(2.0 / math.pi) * (z + 0.044715 * z ** 3)))

def gelu_sig(z):                  # z * sigma(1.702 z), the paper's other approximation
    return z * sigma(1.702 * z)

relu_v = lambda v: np.maximum(v, 0)
print("helpers ready")

helpers ready


## The sentence beat: a hand-set toy neuron
Episode 2's hand-built vectors (concrete thing, living, feeling or state) are illustrative; they are **not** rows of GPT-2's embedding table. One neuron: W = [0, 3, −1] (1 × 3), β = −1. Its output entry is h = W u + β, then the bend ψ.

In [2]:
vec = {"cat": np.array([1, 1, 0]), "mat": np.array([1, 0, 0]), "tired": np.array([0, 0, 1])}
W1 = np.array([0, 3, -1])
beta1 = -1
h_toy = {w: int(W1 @ u + beta1) for w, u in vec.items()}
print("h:", h_toy)
assert h_toy == {"cat": 2, "mat": -1, "tired": -2}

relu_out = [relu(h_toy[w]) for w in ("cat", "mat", "tired")]
print("ReLU:", relu_out)
assert relu_out == [2, 0, 0]

sig_out = [round(sigma(h_toy[w]), 4) for w in ("cat", "mat", "tired")]
gel_out = [round(gelu(h_toy[w]), 4) for w in ("cat", "mat", "tired")]
print("sigma:", sig_out, " GELU:", gel_out)
assert sig_out == [0.8808, 0.2689, 0.1192]
assert gel_out == [1.9545, -0.1587, -0.0455]

h: {'cat': 2, 'mat': -1, 'tired': -2}
ReLU: [2, 0.0, 0.0]
sigma: [0.8808, 0.2689, 0.1192]  GELU: [1.9545, -0.1587, -0.0455]


## Step 1: the collapse in symbols
Layer 1: h = W u + β. Layer 2 reads h: h′ = W′ h + β′. Substituting, h′ = (W′W) u + (W′β + β′): one matrix, one bias.

## Step 2: the collapse in numbers (a 2 × 2 example)

In [3]:
W  = np.array([[1, -1], [2, 1]]);  beta  = np.array([-1, 0])
Wp = np.array([[1, 2], [1, -1]]);  betap = np.array([2, 0])

WpW = Wp @ W
b_merged = Wp @ beta + betap
print("W'W =", WpW.tolist(), "  W'beta + beta' =", b_merged.tolist())
assert WpW.tolist() == [[5, 1], [-1, -2]]
assert b_merged.tolist() == [1, -1]

u = np.array([1, 2])
Wu = W @ u
h = Wu + beta
h2_layers = Wp @ h + betap
h2_merged = WpW @ u + b_merged
print("Wu =", Wu.tolist(), " h =", h.tolist(), " layer by layer:", h2_layers.tolist(), " one matrix:", h2_merged.tolist())
assert Wu.tolist() == [-1, 4] and h.tolist() == [-2, 4]
assert h2_layers.tolist() == [8, -6] == h2_merged.tolist()

# the two forms agree on every integer input in [-5, 5] x [-5, 5] (121 points)
grid = [np.array([a, b]) for a in range(-5, 6) for b in range(-5, 6)]
assert len(grid) == 121
assert all((Wp @ (W @ x + beta) + betap == WpW @ x + b_merged).all() for x in grid)
print("collapse holds on all", len(grid), "integer inputs")

W'W = [[5, 1], [-1, -2]]   W'beta + beta' = [1, -1]
Wu = [-1, 4]  h = [-2, 4]  layer by layer: [8, -6]  one matrix: [8, -6]
collapse holds on all 121 integer inputs


## Step 3: add the bend
Apply ψ = ReLU to each entry between the layers.

In [4]:
def two_layers_with_bend(x):
    return Wp @ relu_v(W @ x + beta) + betap

psi_h = relu_v(h)
out_bent = two_layers_with_bend(u)
print("psi(h) =", psi_h.tolist(), " output with the bend =", out_bent.tolist(), " one-matrix answer =", h2_merged.tolist())
assert psi_h.tolist() == [0, 4]
assert out_bent.tolist() == [10, -4]
assert out_bent.tolist() != h2_merged.tolist()               # the bend changed the output

psi(h) = [0, 4]  output with the bend = [10, -4]  one-matrix answer = [8, -6]


## Step 4: the midpoint test
An affine map f(u) = A u + a satisfies f(u) + f(−u) = 2a = 2 f(0), because A(−u) = −A u cancels A u. So the mean of f(u) and f(−u) must equal f(0).

In [5]:
def collapsed(x):
    return WpW @ x + b_merged

zero = np.array([0, 0])
fu, fmu, f0 = collapsed(u), collapsed(-u), collapsed(zero)
print("collapsed map: f(u) =", fu.tolist(), " f(-u) =", fmu.tolist(), " f(0) =", f0.tolist(), " mean of the first two =", ((fu + fmu) / 2).tolist())
assert fu.tolist() == [8, -6] and fmu.tolist() == [-6, 4] and f0.tolist() == [1, -1]
assert ((fu + fmu) / 2).tolist() == f0.tolist()              # it holds

bu, bmu, b0 = two_layers_with_bend(u), two_layers_with_bend(-u), two_layers_with_bend(zero)
print("with the bend: out(u) =", bu.tolist(), " out(-u) =", bmu.tolist(), " out(0) =", b0.tolist(), " mean of the first two =", ((bu + bmu) / 2).tolist())
assert (W @ (-u)).tolist() == [1, -4]                         # W(-u), as in the episode
assert (W @ (-u) + beta).tolist() == [0, -4] and relu_v(W @ (-u) + beta).tolist() == [0, 0]
assert bu.tolist() == [10, -4] and bmu.tolist() == [2, 0] and b0.tolist() == [2, 0]
assert ((bu + bmu) / 2).tolist() == [6, -2] and ((bu + bmu) / 2).tolist() != b0.tolist()   # the test fails: no affine map matches

collapsed map: f(u) = [8, -6]  f(-u) = [-6, 4]  f(0) = [1, -1]  mean of the first two = [1.0, -1.0]
with the bend: out(u) = [10, -4]  out(-u) = [2, 0]  out(0) = [2, 0]  mean of the first two = [6.0, -2.0]


## Step 5: three bends, four decimals

In [6]:
zs = [-3, -2, -1, 0, 1, 2]
rows = [(z, relu(z), round(gelu(z), 4), round(sigma(z), 4)) for z in zs]
print("  z   ReLU   GELU     sigma")
for r in rows:
    print(f"{r[0]:3d} {r[1]:6.0f} {r[2]:8.4f} {r[3]:8.4f}")
assert [r[1] for r in rows] == [0, 0, 0, 0, 1, 2]
assert [r[2] for r in rows] == [-0.0040, -0.0455, -0.1587, 0.0, 0.8413, 1.9545]
assert [r[3] for r in rows] == [0.0474, 0.1192, 0.2689, 0.5, 0.7311, 0.8808]
assert round(gelu(2), 3) == 1.954 and round(gelu(2), 4) == 1.9545   # four decimals, as the episode says

  z   ReLU   GELU     sigma
 -3      0  -0.0040   0.0474
 -2      0  -0.0455   0.1192
 -1      0  -0.1587   0.2689
  0      0   0.0000   0.5000
  1      1   0.8413   0.7311
  2      2   1.9545   0.8808


## Step 6: GELU forms and slopes
The tanh form and the z·σ(1.702z) form against exact GELU, on a fine grid over [−10, 10]; the slope of σ; and what ten sigmoid slopes multiply to.

In [7]:
erf_v = np.vectorize(math.erf)
zz = np.round(np.arange(-10, 10.0005, 0.001), 3)
gv = zz * 0.5 * (1 + erf_v(zz / math.sqrt(2)))
tv = 0.5 * zz * (1 + np.tanh(math.sqrt(2 / math.pi) * (zz + 0.044715 * zz ** 3)))
sv = zz / (1 + np.exp(-1.702 * zz))

gap_t = np.abs(tv - gv); gap_s = np.abs(sv - gv)
print("tanh form: largest gap %.5f at |z| = %.2f" % (gap_t.max(), abs(zz[gap_t.argmax()])))
print("z*sigma(1.702 z): largest gap %.4f at |z| = %.2f" % (gap_s.max(), abs(zz[gap_s.argmax()])))
assert round(float(gap_t.max()), 5) == 0.00047 and round(abs(float(zz[gap_t.argmax()])), 2) == 2.70
assert round(float(gap_s.max()), 4) == 0.0203 and round(abs(float(zz[gap_s.argmax()])), 2) == 2.27
assert np.allclose(gap_s, gap_s[::-1], atol=1e-12)            # the gap is even in z: same size at +2.27 and -2.27

j = int(gv.argmin())
print("GELU minimum %.4f at z = %.3f" % (gv[j], zz[j]))
assert round(float(gv[j]), 4) == -0.1700 and round(float(zz[j]), 3) == -0.752

# slope of sigma: sigma'(z) = sigma(z)(1 - sigma(z)) = 1/4 - (sigma(z) - 1/2)^2, at most 1/4 at z = 0
sz = 1 / (1 + np.exp(-zz))
dsig = sz * (1 - sz)
assert np.allclose(dsig, 0.25 - (sz - 0.5) ** 2, atol=1e-15)
assert abs(dsig.max() - 0.25) < 1e-12 and abs(zz[dsig.argmax()]) < 1e-9
h_ = 1e-6                                                      # finite differences agree with the formula
for z in (-3.0, -0.5, 0.0, 1.3, 4.0):
    fd = (sigma(z + h_) - sigma(z - h_)) / (2 * h_)
    assert abs(fd - sigma(z) * (1 - sigma(z))) < 1e-9

p10 = 0.25 ** 10
print("(1/4)^10 =", p10)
assert round(p10 * 1e7, 2) == 9.54 and 9.53e-7 < p10 < 9.54e-7   # 'about 9.54 x 10^-7', one factor of the chain rule only

# softmax of the pair (z, 0) has first entry sigma(z)
for z in (-3.0, 0.0, 2.5):
    e = np.exp([z, 0.0]); assert abs(e[0] / e.sum() - sigma(z)) < 1e-15

tanh form: largest gap 0.00047 at |z| = 2.70
z*sigma(1.702 z): largest gap 0.0203 at |z| = 2.27
GELU minimum -0.1700 at z = -0.752
(1/4)^10 = 9.5367431640625e-07


## Check question
Use the Math Lab's W, β, W′, β′. (a) The one-matrix map at u = (0, 0). (b) The network with ReLU between the layers at u = (0, 0). (c) For the 11 tokens, the shape leaving the first feed-forward layer and the number of GELUs in one block.

In [8]:
assert collapsed(zero).tolist() == [1, -1]                    # (a) the bias W'beta + beta'
assert two_layers_with_bend(zero).tolist() == [2, 0]          # (b) h = beta = (-1, 0), psi(h) = (0, 0), output beta'
n_tokens, width_in, width_out = 11, 768, 3072
shape_after = (n_tokens, width_out)                           # (11 x 768)(768 x 3,072) = 11 x 3,072
print("(a) (1, -1)   (b) (2, 0)   (c)", shape_after, "and", n_tokens * width_out, "GELUs per block")
assert shape_after == (11, 3072) and n_tokens * width_out == 33_792

(a) (1, -1)   (b) (2, 0)   (c) (11, 3072) and 33792 GELUs per block


## Real scale: GPT-2 small's feed-forward sub-layer
Each of the 12 blocks has a feed-forward sub-layer: a linear layer from 768 to 3,072 numbers, GELU on each of the 3,072, and a linear layer back to 768. This cell downloads the published weights (Hugging Face `openai-community/gpt2`, 548 MB, once) and counts. Hugging Face stores GPT-2's linear weights as [in, out], the transpose of Episode 3's W. (This checkpoint was not compared with OpenAI's original release.) Skip this section on a slow connection.

In [9]:
import subprocess, sys
for pkg in ["safetensors", "huggingface_hub"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
from huggingface_hub import hf_hub_download
from safetensors import safe_open

path = hf_hub_download("openai-community/gpt2", "model.safetensors", local_dir="gpt2_weights")
f = safe_open(path, framework="np")
shape = lambda name: tuple(f.get_slice(name).get_shape())
size = lambda name: int(np.prod(shape(name)))

assert shape("h.0.mlp.c_fc.weight") == (768, 3072) and shape("h.0.mlp.c_fc.bias") == (3072,)
assert shape("h.0.mlp.c_proj.weight") == (3072, 768) and shape("h.0.mlp.c_proj.bias") == (768,)

per_block = [sum(size(f"h.{b}.mlp.{p}") for p in ("c_fc.weight", "c_fc.bias", "c_proj.weight", "c_proj.bias")) for b in range(12)]
assert set(per_block) == {4_722_432}
ffn_total = sum(per_block)
fc = 768 * 3072 + 3072
pr = 3072 * 768 + 768
print("c_fc", fc, "+ c_proj", pr, "=", fc + pr, "per block;", ffn_total, "over 12 blocks")
assert (fc, pr, fc + pr, ffn_total) == (2_362_368, 2_360_064, 4_722_432, 56_669_184)

# total parameters (the 12 attention-mask buffers h.N.attn.bias are not parameters)
total_params = sum(size(k) for k in f.keys() if not k.endswith(".attn.bias"))
wte = size("wte.weight")
print("parameters", total_params, "; feed-forward share %.1f%%" % (100 * ffn_total / total_params), "; embedding table share %.1f%%" % (100 * wte / total_params))
assert total_params == 124_439_808 and wte == 38_597_376
assert round(100 * ffn_total / total_params, 1) == 45.5 and round(100 * wte / total_params, 1) == 31.0
print("ALL EPISODE 11 CHECKS PASSED")

c_fc 2362368 + c_proj 2360064 = 4722432 per block; 56669184 over 12 blocks
parameters 124439808 ; feed-forward share 45.5% ; embedding table share 31.0%
ALL EPISODE 11 CHECKS PASSED
